# Personal Finance Modeling Notebook (Two-Phase)

This notebook is organized in two phases:

1. **Phase 1**: Original Ridge regression model predicting `PWNETWPG` (net worth)
2. **Phase 2**: Financial Resilience Score (FRS) framework + stress testing

All cells are executable and this notebook is saved with outputs.


In [1]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'analysis' / 'run_finance_ml.py').exists():
    PROJECT_ROOT = Path('/Users/lincolncheng/CS Project/SSDC_Datathon')

OUTPUT_DIR = PROJECT_ROOT / 'outputs'
XLSX_PATH = PROJECT_ROOT / 'personal_finance_dataset.xlsx'

print('Project root:', PROJECT_ROOT)
print('Python executable:', sys.executable)
print('Dataset exists:', XLSX_PATH.exists())


Project root: /Users/lincolncheng/CS Project/SSDC_Datathon
Python executable: /Users/lincolncheng/CS Project/SSDC_Datathon/.venv/bin/python
Dataset exists: True


## Run Pipeline (Generates Both Phases)

In [2]:
cmd = [
    sys.executable,
    str(PROJECT_ROOT / 'analysis' / 'run_finance_ml.py'),
    '--xlsx', str(XLSX_PATH),
    '--outdir', str(OUTPUT_DIR),
    '--seed', '42',
]

result = subprocess.run(cmd, capture_output=True, text=True)
print('Return code:', result.returncode)
print(result.stdout.strip())
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError('Pipeline execution failed')


Return code: 0
Finished. Outputs written to: /Users/lincolncheng/CS Project/SSDC_Datathon/outputs


In [3]:
metrics = json.loads((OUTPUT_DIR / 'metrics.json').read_text())
factor_loadings = pd.read_csv(OUTPUT_DIR / 'factor_loadings.csv')
scenario_summary = pd.read_csv(OUTPUT_DIR / 'resilience_scenario_summary.csv')
tier_summary = pd.read_csv(OUTPUT_DIR / 'resilience_tier_summary.csv')
cluster_resilience = pd.read_csv(OUTPUT_DIR / 'resilience_cluster_summary.csv')

print('Files in outputs:')
for p in sorted(OUTPUT_DIR.iterdir()):
    print('-', p.name)


Files in outputs:
- analysis_summary.md
- anomalies.csv
- cluster_profiles.csv
- dictionary_cleaned.csv
- engineered_dataset_with_cluster.csv
- factor_loadings.csv
- metrics.json
- resilience_bootstrap_stability.json
- resilience_cluster_summary.csv
- resilience_scenario_summary.csv
- resilience_scores.csv
- resilience_tier_summary.csv
- resilience_transition_matrix.csv


## Phase 1: Original Net-Worth Model (Ridge Regression)

In [4]:
phase1 = pd.DataFrame([
    {
        'target': metrics['dataset']['target'],
        'best_alpha': metrics['model']['best_alpha'],
        'train_r2': metrics['model']['train_r2'],
        'test_r2': metrics['model']['test_r2'],
        'train_mae': metrics['model']['train_mae'],
        'test_mae': metrics['model']['test_mae'],
        'train_rmse': metrics['model']['train_rmse'],
        'test_rmse': metrics['model']['test_rmse'],
    }
])
phase1


,target,best_alpha,train_r2,test_r2,train_mae,test_mae,train_rmse,test_rmse
0,PWNETWPG,25.0,0.7037,0.693909,586137.157471,605004.638418,960682.528031,1.008757e+06


In [5]:
cv_alpha = pd.DataFrame([
    {'alpha': float(k), 'cv_r2': v}
    for k, v in metrics['model']['cv_r2_by_alpha'].items()
]).sort_values('alpha')
cv_alpha


,alpha,cv_r2
0,0.1,0.699653
1,1.0,0.699696
2,5.0,0.699726
3,10.0,0.699744
4,25.0,0.699759
5,50.0,0.699734
6,100.0,0.699648
7,250.0,0.699391
8,500.0,0.698888
9,1000.0,0.697449


In [6]:
print('Top positive drivers from Phase 1 model:')
display(factor_loadings.sort_values('beta_std', ascending=False).head(12))

print('Top negative drivers from Phase 1 model:')
display(factor_loadings.sort_values('beta_std', ascending=True).head(12))


Top positive drivers from Phase 1 model:


,feature,beta_std,abs_beta_std,readable_feature
0,HOME_EQUITY,551232.616641,551232.616641,Home Equity (Home Value - Mortgage)
1,PEFATINC,429156.063965,429156.063965,After-Tax Income
2,PWAPRVAL,342093.368057,342093.368057,Home Value
3,PFMTYPG=9,340630.403410,340630.403410,Family Type
4,SAVINGS_GAP,334266.059881,334266.059881,Savings Minus Credit Card Debt
5,LIQUID_ASSETS,313813.208608,313813.208608,Liquid Assets (Deposits + TFSA)
8,DEBT_TO_INCOME,100556.697978,100556.697978,Debt-to-Income Ratio
11,PAGEMIEG=5,76264.807644,76264.807644,Age Group
12,TOTAL_DEBT,71311.976087,71311.976087,Total Debt (Mortgage + Student + Credit Card +...
16,PAGEMIEG=6,61452.132584,61452.132584,Age Group


Top negative drivers from Phase 1 model:


,feature,beta_std,abs_beta_std,readable_feature
6,PWDPRMOR,-240805.127205,240805.127205,Mortgage Debt
7,PWASTDEP,-211439.309283,211439.309283,Bank Deposits
9,PAGEMIEG=2,-95413.381593,95413.381593,Age Group
10,PFMTYPG=4,-87230.016449,87230.016449,Family Type
13,PFMTYPG=3,-67687.162116,67687.162116,Family Type
14,PFMTYPG=2,-65031.463259,65031.463259,Family Type
15,PPVRES=11,-64091.525429,64091.525429,Province
17,LIQUIDITY_TO_INCOME,-58395.423956,58395.423956,Liquidity-to-Income Ratio
18,PPVRES=10,-54591.591407,54591.591407,Province
19,PPVRES=59,-54486.433220,54486.433220,Province


## Phase 2: Financial Resilience Score (FRS)

In [7]:
res = metrics['resilience']
phase2_overview = pd.DataFrame([
    {
        'frs_q20': res['tier_thresholds']['q20'],
        'frs_q50': res['tier_thresholds']['q50'],
        'frs_q80': res['tier_thresholds']['q80'],
        'corr_frs_net_worth': res['validation']['corr_frs_vs_net_worth'],
        'corr_frs_anomaly_distance': res['validation']['corr_frs_vs_mahalanobis_sq'],
        'frs_only_net_worth_r2': res['validation']['frs_only_regression']['r2'],
    }
])
phase2_overview


,frs_q20,frs_q50,frs_q80,corr_frs_net_worth,corr_frs_anomaly_distance,frs_only_net_worth_r2
0,38.719,48.046303,63.839181,0.415653,0.316921,0.172767


In [8]:
print('Baseline FRS tiers:')
tier_summary


Baseline FRS tiers:


,FRS_TIER_BASELINE,count,share,PEFATINC,PWNETWPG,TOTAL_DEBT,LIQUID_ASSETS,DEBT_TO_INCOME,LIQUIDITY_TO_INCOME,HOME_EQUITY_TO_VALUE,SAVINGS_GAP,PWDSTCRD
0,At Risk,3249,0.2,88750.0,477200.0,212500.0,5000.0,2.4226,0.0585,0.4875,0.0,7000.0
1,Coping,4872,0.3,63375.0,412050.0,7250.0,5250.0,0.1265,0.0875,0.8619,4750.0,0.0
2,Stable,4872,0.3,98150.0,1248375.0,0.0,59000.0,0.0000,0.6721,1.0000,58500.0,0.0
3,Thriving,3248,0.2,94662.5,2311000.0,0.0,245000.0,0.0000,2.8198,1.0000,245000.0,0.0


In [9]:
print('Stress scenario summary:')
scenario_summary


Stress scenario summary:


,scenario,mean_frs_post,mean_frs_drop,median_frs_drop,pct_below_baseline_fragility_cutoff,most_impacted_cluster,most_impacted_cluster_mean_drop,share_at_risk_post,share_coping_post,share_stable_post,share_thriving_post
0,rate_hike_200bp,50.893736,0.137040,0.007780,0.204852,0,0.472372,0.204852,0.297888,0.297888,0.199372
1,income_shock_20pct,50.955969,0.074807,0.351246,0.221107,0,1.609675,0.221107,0.297026,0.269503,0.212364
2,housing_shock_15pct,50.745952,0.284824,0.000000,0.212302,0,0.855477,0.212302,0.292778,0.295671,0.199249


In [10]:
print('Cluster-level resilience breakdown:')
cluster_resilience


Cluster-level resilience breakdown:


,CLUSTER,count,frs_mean,frs_median,share_dataset,tier_share_At Risk,tier_share_Coping,tier_share_Stable,tier_share_Thriving
0,0,2982,37.1274,36.0282,0.1836,0.5949,0.2243,0.1392,0.0416
1,1,13259,54.1577,50.0748,0.8164,0.1112,0.3170,0.3361,0.2356


In [11]:
transition = pd.read_csv(OUTPUT_DIR / 'resilience_transition_matrix.csv')
transition.head(24)


,baseline_tier,scenario_tier,count,share_within_baseline_tier,scenario
0,At Risk,At Risk,3249,1.000000,rate_hike_200bp
1,At Risk,Coping,0,0.000000,rate_hike_200bp
2,At Risk,Stable,0,0.000000,rate_hike_200bp
3,At Risk,Thriving,0,0.000000,rate_hike_200bp
4,Coping,At Risk,78,0.016010,rate_hike_200bp
5,Coping,Coping,4794,0.983990,rate_hike_200bp
6,Coping,Stable,0,0.000000,rate_hike_200bp
7,Coping,Thriving,0,0.000000,rate_hike_200bp
8,Stable,At Risk,0,0.000000,rate_hike_200bp
9,Stable,Coping,44,0.009031,rate_hike_200bp


## Presentation Note

Use this storyline in slides:

- **Phase 1:** Predict net worth (`PWNETWPG`) and explain feature drivers.
- **Phase 2:** Move from wealth prediction to shock resilience using FRS tiers and stress transitions.
